# Assignment 5: Comparing estimators for one policy date
**Econ 521 · 100 points**

Two applied questions, 50 points each. Both ask the same thing in different settings: **when the estimators disagree, which counterfactual do you believe, and why?**

- **Question 1** uses **castle-doctrine laws** (Cheng & Hoekstra 2013), the staggered event study from the 22 Oct lecture. You compare the TWFE event study, a "clean" cohort-by-cohort event study, and an interrupted time series for a single state.
- **Question 2** uses **German reunification** (Abadie, Diamond & Hainmueller 2015, *AJPS*). You compare ITS, DiD, synthetic control and synthetic DiD for West Germany, then test how fragile each is.

**How this relates to the labs.** Lab 11 computed ITS, DiD, SC and SDID on Prop 99. You may copy its solver (`proj_simplex`, `simplex_weights`). None of the analyses below was done in a lab: staggered timing, the cohort-by-cohort estimator, single-state ITS against a comparison group, a case where DiD has the **wrong sign**, and leave-one-out robustness are new.

**Grading.** Points reward (1) a design **stated before** you see results, (2) justification, (3) correct implementation, and (4) honest interpretation. You are **not** graded on matching a key number. Respect the word limits; text beyond them is not read.

**Files** (next to this notebook or in `data/`): `castle.dta`, `repgermany.csv`. Allowed packages: numpy, pandas, scipy, statsmodels, matplotlib. Plot convention: blue = treated, orange = comparison or synthetic, never color alone.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from pathlib import Path

BLUE, ORANGE, GRAY = "#1f77b4", "#ff7f0e", "#7f7f7f"
plt.rcParams.update({"figure.figsize": (8, 4.2), "axes.spines.top": False, "axes.spines.right": False})

def find(name):
    for p in [Path(name), Path("data") / name]:
        if p.exists():
            return p
    raise FileNotFoundError(name)

castle = pd.read_stata(find("castle.dta"))
germany = pd.read_csv(find("repgermany.csv"))
print(castle.shape, germany.shape)

---
## Question 1 · Castle doctrine: three ways to draw the counterfactual (50 points)

**Setup.** 50 states, 2000–2010. `effyear` is the year the law took effect (missing if never adopted). Outcome: `l_homicide`, the log homicide rate. Cohorts: 2005 (1 state, Florida), 2006 (13), 2007 (4), 2008 (2), 2009 (1); 29 states never adopt. No controls and no weights throughout.

**(a) (6 pts) Design statement, written before you run anything (≤ 120 words).** State: the reference period; the event-time window you will **report**; the comparison group for (c); and the size of a pre-period coefficient that would make you doubt parallel trends, judged against what? (Lecture: requirement 2, "the reference period is chosen first".)

**Answer (a):**

**(b) (10 pts) TWFE event study.** Estimate
$$Y_{it} = \alpha_i + \lambda_t + \sum_{k \ne -1} \delta_k\, 1[t - E_i = k] + \varepsilon_{it}$$
with standard errors clustered by state (never-adopters get no event-time dummy). Report: (i) the coefficients and SEs in a table that also shows **how many states identify each $k$**; (ii) the event-study plot with 95% CIs; (iii) the single-coefficient TWFE estimate using `cdl`. *Note:* statsmodels and R's `fixest` use slightly different small-sample corrections, so SEs differ from the lecture's in the second decimal. That is fine.

In [ ]:
# Your code here


**(c) (14 pts) A clean event study, by hand.** With staggered timing, TWFE also compares newly treated states with **already treated** states (lecture slide 14; Lab 8). Build the alternative directly from 2×2 comparisons against never-treated states:

1. For each cohort $g$ and year $t \ne g-1$: $\widehat{ATT}(g,t) = (\bar Y_{g,t} - \bar Y_{g,g-1}) - (\bar Y_{\infty,t} - \bar Y_{\infty,g-1})$, where $\infty$ denotes the never-treated states. (Use the cohort's last pre-year as the base for every $t$.)
2. Aggregate by event time $k = t-g$, weighting cohorts by their number of states.
3. Overall post-period ATT: the state-weighted average of all $\widehat{ATT}(g,t)$ with $t \ge g$.

Plot the clean path next to the TWFE path from (b) on one figure. In ≤ 120 words: where do they differ, and why are the differences **small** in this application? (*Hint:* how many states are ever used as "controls" after being treated, and how large is the never-treated group?)

In [ ]:
# Your code here


**(d) (12 pts) One state, no comparison group.** Florida is the only 2005 adopter. Using Florida's series alone, estimate the ITS regression from the lecture with $t_0 = 2005$ and Newey–West SEs (2 lags):
$$Y_t = \alpha + \beta (t-t_0) + \gamma D_t + \delta D_t (t-t_0) + \varepsilon_t.$$
Then compute Florida's DiD against the never-treated states: average of $(Y_{FL,t} - Y_{FL,2004}) - (\bar Y_{\infty,t} - \bar Y_{\infty,2004})$ over 2005–2010. Plot Florida with the ITS fitted lines and the extended pre-line. In ≤ 120 words: what does each estimate assume, how many pre-period observations support the ITS line, and which do you trust more for Florida?

In [ ]:
# Your code here


**(e) (8 pts) Report (≤ 150 words).** You can show a policymaker one plot and one number. Which do you choose from (b)–(d), and why? Address the far leads, the number of states behind the last lag, and what your number does **not** tell them.

**Answer (e):**

---
## Question 2 · German reunification: when the estimators disagree on the sign (50 points)

**Setup.** `repgermany.csv`: 17 OECD countries, 1960–2003. Outcome: `gdp`, GDP per capita (PPP, 2002 USD). Treated unit: **West Germany**. Reunification: 3 October 1990, so the first treated year is **1990** and the pre-period is 1960–1989. Donors: the other 16 countries. Abadie, Diamond & Hainmueller (2015) use covariates and a $V$ matrix. You will use **outcome-only** estimators, as in Lab 11.

**(a) (6 pts) Design memo, before estimating (≤ 150 words).** (i) Is 1990 the right first treated year, or could effects anticipate it (the Wall fell in November 1989)? (ii) Name **two** donors that might themselves be affected by reunification, and say whether you keep them. (iii) Before looking at the data: do you expect West Germany to have grown **faster or slower** than the average donor before 1990, and what would that do to a DiD?

**Answer (a):**

**(b) (8 pts) ITS and the event-study leads.** (i) Estimate the ITS regression for West Germany alone with $t_0 = 1990$ (Newey–West, 2 lags) and report the average gap between actual GDP and the extended pre-line over 1990–2003. (ii) Compute the one-treated-unit event study against the **equal-weighted** average donor with reference year **1989** and plot it. Report the leads for 1960 and 1975. In ≤ 80 words: what do the leads say about DiD?

In [ ]:
# Your code here


**(c) (14 pts) DiD, SC and SDID.** Compute the three estimators exactly as in Lab 11 (DiD with equal weights; outcome-only SC; SDID with $\zeta = (T_{post})^{1/4}\hat\sigma$). Report: (i) a table with the estimate, the pre-period RMSE of the gap, and the number of donors with weight > 0.001; (ii) the SC weights and the SDID time weights; (iii) one figure with the four gap paths (ITS, DiD, SC, SDID). In ≤ 120 words: why does DiD give a **positive** estimate while SC and SDID give negative ones? Use the weights and the leads.

In [ ]:
# Your code here


**(d) (10 pts) Leave-one-out.** For each donor with positive SC weight, drop it from the donor pool and recompute SC **and** SDID (keep everything else fixed). Report the range of each estimator across the leave-one-out runs and plot the SC gap paths (thin lines) with the full-sample path (thick). In ≤ 100 words: which donor matters most, and is either estimator's conclusion fragile?

In [ ]:
# Your code here


**(e) (12 pts) Memo to a policymaker (≤ 200 words).** A minister asks: "Did reunification make West Germans poorer, and by how much?" Using (b)–(d), answer, say which estimator you report and why, explain in plain language why the DiD and ITS numbers should **not** be used here, and state the main remaining threat (think of (a)(ii)).

**Answer (e):**